In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt

# 1. Cargar shapefile de distritos
dpt_shp = gpd.read_file(r'C:\\Users\\HP\\OneDrive - Universidad del Pacífico\\textos\\Escritorio\\2025-1\\Git\\PC\\PC3\\shape_file\\DISTRITOS.shp')
dpt_shp["DISTRITO"] = dpt_shp["DISTRITO"].str.upper()

# 2. Cargar archivo de colegios
final_df = pd.read_excel(r'C:\\Users\\HP\\OneDrive - Universidad del Pacífico\\textos\\Escritorio\\2025-1\\Git\\PC\\PC3\\archivo_final.xlsx')

# 3. Normalizar texto
final_df["Distrito"] = final_df["Distrito"].str.upper()
final_df["Departamento"] = final_df["Departamento"].str.upper()
final_df["Nivel"] = final_df["Nivel / Modalidad"].str.upper()


In [ ]:
# Niveles educativos que quieres mapear
niveles = ["INICIAL", "PRIMARIA", "SECUNDARIA"]

for nivel in niveles:
    # Filtrar colegios por nivel
    df_nivel = final_df[final_df["Nivel"].str.contains(nivel)]

    # Agrupar por distrito
    conteo = df_nivel.groupby("Distrito").size().reset_index(name="n_escuelas")

    # Unir shapefile con datos de conteo
    mapa = dpt_shp.merge(conteo, left_on="DISTRITO", right_on="Distrito", how="left")
    mapa["n_escuelas"] = mapa["n_escuelas"].fillna(0)

    # Crear mapa estático
    ax = mapa.plot(
        column="n_escuelas",
        cmap="Blues",
        legend=True,
        edgecolor="black",
        figsize=(12, 8)
    )
    plt.title(f"Distribución de Escuelas - Nivel {nivel.title()}")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(f"mapa_{nivel.lower()}.png")
    plt.close()


In [ ]:
# Filtrar solo Huancavelica y Ayacucho
df_hua_aya = final_df[final_df["Departamento"].isin(["HUANCAVELICA", "AYACUCHO"])]

# Filtrar primarias y secundarias
primarias = df_hua_aya[df_hua_aya["Nivel"].str.contains("PRIMARIA")]
secundarias = df_hua_aya[df_hua_aya["Nivel"].str.contains("SECUNDARIA")]

# Crear GeoDataFrames a partir de latitud y longitud
primarias_gdf = gpd.GeoDataFrame(primarias, geometry=gpd.points_from_xy(primarias["Longitud"], primarias["Latitud"]), crs="EPSG:4326")
secundarias_gdf = gpd.GeoDataFrame(secundarias, geometry=gpd.points_from_xy(secundarias["Longitud"], secundarias["Latitud"]), crs="EPSG:4326")

# Convertir a sistema métrico (zona UTM 18S)
primarias_gdf = primarias_gdf.to_crs(epsg=32718)
secundarias_gdf = secundarias_gdf.to_crs(epsg=32718)

# Crear buffer de 5 km y contar secundarias cercanas
conteos = []
for idx, primaria in primarias_gdf.iterrows():
    buffer = primaria.geometry.buffer(5000)  # 5 km
    cercanas = secundarias_gdf[secundarias_gdf.geometry.within(buffer)]
    conteos.append(len(cercanas))

# Añadir columna de conteo
primarias_gdf["SecundariasCerca"] = conteos

# Identificar extremos
min_idx = primarias_gdf["SecundariasCerca"].idxmin()
max_idx = primarias_gdf["SecundariasCerca"].idxmax()

# Ver resultado
print("Primaria con MENOS secundarias cerca:", primarias_gdf.loc[min_idx]["Nombre de SS.EE."], "→", primarias_gdf.loc[min_idx]["SecundariasCerca"])
print("Primaria con MÁS secundarias cerca:", primarias_gdf.loc[max_idx]["Nombre de SS.EE."], "→", primarias_gdf.loc[max_idx]["SecundariasCerca"])


In [ ]:
# Visualizar en matplotlib los buffers
fig, ax = plt.subplots(figsize=(12, 10))

# Dibujar todas las secundarias
secundarias_gdf.plot(ax=ax, color='blue', markersize=5, label='Secundarias')

# Dibujar los buffers de las dos primarias seleccionadas
primarias_gdf.loc[[min_idx, max_idx]].geometry.buffer(5000).plot(
    ax=ax,
    edgecolor='red',
    facecolor='none',
    linestyle='--',
    linewidth=1,
    label='Buffer 5 km'
)

# Dibujar las primarias seleccionadas
primarias_gdf.loc[[min_idx, max_idx]].plot(
    ax=ax,
    color='green',
    markersize=50,
    label='Primarias seleccionadas'
)

plt.legend()
plt.title("Proximidad entre primarias y secundarias - Huancavelica y Ayacucho")
plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
import folium
from folium.plugins import MarkerCluster

# Reproyectar a EPSG:4326 (Folium solo acepta coordenadas lat/lon)
primarias_gdf = primarias_gdf.to_crs(epsg=4326)
secundarias_gdf = secundarias_gdf.to_crs(epsg=4326)

# Crear mapa centrado en Huancavelica
m = folium.Map(location=[-13.0, -74.0], zoom_start=7)

# Añadir marcador para primaria con más secundarias cercanas
folium.Marker(
    location=[mas_cercanas.geometry.y, mas_cercanas.geometry.x],
    popup=f"{mas_cercanas['Nombre de SS.EE.']} (más cercanas: {mas_cercanas['SecundariasCerca']})",
    icon=folium.Icon(color='green', icon='school', prefix='fa')
).add_to(m)

# Añadir marcador para primaria con menos secundarias cercanas
folium.Marker(
    location=[menos_cercanas.geometry.y, menos_cercanas.geometry.x],
    popup=f"{menos_cercanas['Nombre de SS.EE.']} (menos cercanas: {menos_cercanas['SecundariasCerca']})",
    icon=folium.Icon(color='red', icon='school', prefix='fa')
).add_to(m)

# Añadir círculos (5 km)
folium.Circle(
    radius=5000,
    location=[mas_cercanas.geometry.y, mas_cercanas.geometry.x],
    color='green',
    fill=False,
    tooltip="Buffer 5 km"
).add_to(m)

folium.Circle(
    radius=5000,
    location=[menos_cercanas.geometry.y, menos_cercanas.geometry.x],
    color='red',
    fill=False,
    tooltip="Buffer 5 km"
).add_to(m)

# Añadir todas las secundarias como puntos azules
marker_cluster = MarkerCluster().add_to(m)
for _, row in secundarias_gdf.iterrows():
    folium.CircleMarker(
        location=(row.geometry.y, row.geometry.x),
        radius=2,
        color="blue",
        fill=True,
        fill_opacity=0.7
    ).add_to(marker_cluster)

# Guardar como HTML
m.save("mapa_proximidad_interactivo.html")


In [ ]:
import streamlit as st
import pandas as pd
import base64

st.set_page_config(layout="wide")

# ---------- TAB 1 ----------
st.title("Análisis Geoespacial de Escuelas en Perú")

tab1, tab2, tab3 = st.tabs(["📊 Descripción de Datos", "🗺️ Mapas Estáticos", "🌐 Mapas Interactivos"])

with tab1:
    st.subheader("Datos generales")
    st.markdown("Archivo fuente: `archivo_final.xlsx` con coordenadas y características de cada institución educativa.")
    st.markdown("**Departamentos analizados para proximidad:** Huancavelica y Ayacucho.")
    st.image("grafico_barras_resumen.png")  # si haces un gráfico resumen

# ---------- TAB 2 ----------
with tab2:
    st.subheader("Distribución de Escuelas por Nivel")
    col1, col2, col3 = st.columns(3)
    with col1:
        st.image("mapa_inicial.png", caption="Nivel Inicial")
    with col2:
        st.image("mapa_primaria.png", caption="Nivel Primaria")
    with col3:
        st.image("mapa_secundaria.png", caption="Nivel Secundaria")

# ---------- TAB 3 ----------
with tab3:
    st.subheader("Mapa de Proximidad Interactivo")
    with open("mapa_proximidad_interactivo.html", "r", encoding="utf-8") as f:
        folium_html = f.read()
    st.components.v1.html(folium_html, height=600, scrolling=True)
